# 08 · Bounds and Constraints

Smoothing keeps slip smooth, but it cannot stop the inversion from producing
*physically impossible* slip — negative (back-) slip on a fault that only moved
one way, or amplitudes larger than geology allows. Those are **inequality**
priors, which quadratic regularization cannot express. This notebook adds them.

## Learning objectives
- Enforce non-negative slip with **NNLS**.
- Apply general lower/upper bounds with **bounded least squares**.
- Add linear **inequality constraints** as a quadratic program.
- Reduce parameters by fixing slip direction (`components='rake'`/`'azimuth'`).

## Priors that regularization can't express

Regularization penalizes *how much* a model is disliked (a quadratic cost).
Some priors are **hard** and one-sided:

- slip should not reverse sense (no back-slip): $\mathbf m \ge 0$;
- slip magnitude is capped: $\mathbf m \le u$;
- more general geologic limits: $C\mathbf m \le \mathbf d_{\text{ineq}}$.

`geodef.invert()` handles these through the `bounds` and `constraints` arguments
and picks the right solver automatically:

- `bounds=(0, None)` → **non-negative least squares (NNLS)**;
- `bounds=(lb, ub)` → **bounded least squares**;
- `method='constrained', constraints=(C, d)` → a **quadratic program (QP)**
  with $C\mathbf m \le \mathbf d$.

Bounds may be a scalar, a per-component pair, or a full per-parameter array.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## Non-negative slip

A lightly-smoothed unconstrained inversion dips slightly *negative* at the edges
— spurious back-slip. Requiring $\mathbf m \ge 0$ removes it. Because the true
slip is one-signed, this is a physically correct prior.

In [2]:
kw = dict(components="dip", smoothing="laplacian", smoothing_strength=0.2)
free = geodef.invert(fault, gnss, **kw)                    # can go negative
nn = geodef.invert(fault, gnss, bounds=(0, None), **kw)    # auto-NNLS

print(f"unconstrained min slip: {free.slip_vector.min()*100:+.1f} cm (spurious back-slip)")
print(f"non-negative  min slip: {nn.slip_vector.min()*100:+.1f} cm")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
lim = abs(free.slip_vector).max()
geodef.plot.slip(fault, free.slip_vector, ax=axes[0], cmap="RdBu_r",
                 vmin=-lim, vmax=lim, title="Unconstrained", colorbar_label="Slip (m)")
geodef.plot.slip(fault, nn.slip_vector, ax=axes[1], cmap="RdBu_r",
                 vmin=-lim, vmax=lim, title="Non-negative (NNLS)", colorbar_label="Slip (m)")
plt.tight_layout()

unconstrained min slip: -36.3 cm (spurious back-slip)
non-negative  min slip: +0.0 cm


## Bounded least squares

Bounds can be two-sided. Cap the slip at 1 m to see where the upper bound starts
to bite (and bias the fit).

In [3]:
capped = geodef.invert(fault, gnss, bounds=(0, 1.0), **kw)
print(f"max slip, uncapped: {nn.slip_vector.max():.2f} m")
print(f"max slip, capped:   {capped.slip_vector.max():.2f} m (hits the 1.0 m ceiling)")
print(f"reduced chi^2, uncapped {nn.reduced_chi2:.2f}  vs capped {capped.reduced_chi2:.2f}")

max slip, uncapped: 1.76 m
max slip, capped:   1.00 m (hits the 1.0 m ceiling)
reduced chi^2, uncapped 1.04  vs capped 2.25


## Fixed slip direction

Another way to encode a prior is to fix the *direction* of slip and solve for one
amplitude per patch. `components='rake'` fixes a single rake in each patch's
local strike-dip frame; `components='azimuth'` fixes a geographic slip azimuth
(correct even when strike varies). This halves the unknowns and cleanly enforces
a sense of motion.

In [4]:
# rake = 90 deg is pure dip-slip in the local frame (matches our true slip).
rk = geodef.invert(fault, gnss, components="rake", rake=90.0,
                   smoothing="laplacian", smoothing_strength=1.0)
print(f"components='rake' solves {rk.slip_vector.size} amplitudes "
      f"(vs {2*N} for 'both')")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True", colorbar_label="Slip (m)")
geodef.plot.slip(fault, rk.slip_vector, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="Fixed-rake amplitude",
                 colorbar_label="Slip (m)")
plt.tight_layout()

components='rake' solves 40 amplitudes (vs 80 for 'both')


## Exercises
1. **Three-way compare.** Plot WLS, NNLS, and the fixed-rake solution side by
   side on the same data. Which is smoothest? Which best matches the truth?
2. **Upper bound.** Lower the cap from `1.0` toward `0.5` m. At what value does
   the reduced $\chi^2$ start to climb noticeably (the bound begins to bias the fit)?
3. **Per-component bounds.** With `components='both'`, pass
   `bounds=(np.array([0.0, 0.0]), None)` to force *both* slip components
   non-negative, and compare to the unconstrained both-component result.

## Checkpoint questions
1. Why can't quadratic regularization enforce $\mathbf m \ge 0$?
2. What does fixing the rake buy you, in terms of unknowns and priors?
3. When does an upper bound help, and when does it just bias the fit?

## Summary
- Inequality priors need bounds or constraints, not just smoothing.
- `bounds=(0, None)` → NNLS; `bounds=(lb, ub)` → bounded LS;
  `constraints=(C, d)` → QP.
- Fixing slip direction (`'rake'`/`'azimuth'`) halves the unknowns and encodes a sense of slip.

**Next:** notebook 09 asks how well the slip is actually *resolved* —
uncertainty and resolution.